In [1]:
import tarfile

filename = "phemernrdataset.tar.bz2"

# Open the file
with tarfile.open(filename, "r:bz2") as tar:
    # Extract all contents to the current directory
    tar.extractall(path="./extracted_data")
    print("Extraction complete.")

C:\Users\jacki\AppData\Local\Temp\ipykernel_39096\4018046509.py:8: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path="./extracted_data")


Extraction complete.


In [ ]:
import json
import csv
from pathlib import Path

ROOT = Path("./extracted_data/pheme-rnr-dataset")
OUT_CSV = Path("./pheme_reactions_3col.csv")

def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def get_label_from_path(p: Path) -> int:
    parts = set(p.parts)
    if "rumours" in parts:
        return 1
    if "non-rumours" in parts:
        return 0
    # If it doesn't match either, skip (or you can raise an error)
    return None

def get_source_text_for_thread(thread_dir: Path) -> str:
    source_dir = thread_dir / "source-tweet"
    if not source_dir.exists():
        return ""

    # Usually there is exactly one source tweet json in this folder.
    source_files = sorted(source_dir.glob("*.json"))
    if not source_files:
        return ""

    source_obj = load_json(source_files[0])
    return source_obj.get("text", "") or ""

def main():
    reaction_files = list(ROOT.rglob("reactions/*.json"))

    with OUT_CSV.open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["text", "label", "source_text"])

        kept = 0
        skipped = 0

        for rf in reaction_files:
            label = get_label_from_path(rf)
            if label is None:
                skipped += 1
                continue

            try:
                reaction_obj = load_json(rf)
            except Exception:
                skipped += 1
                continue

            reaction_text = reaction_obj.get("text", "")
            if not reaction_text:
                skipped += 1
                continue

            # rf is .../<thread_id>/reactions/<tweet_id>.json
            # so thread_dir is parent of "reactions"
            thread_dir = rf.parent.parent
            source_text = get_source_text_for_thread(thread_dir)

            writer.writerow([reaction_text, label, source_text])
            kept += 1

    print(f"Done! Wrote: {OUT_CSV}")
    print(f"Rows written: {kept}")
    print(f"Skipped: {skipped}")

if __name__ == "__main__":
    main()

Done! Wrote: pheme_reactions_3col.csv
Rows written: 97410
Skipped: 0
